# Perfect Team app audit — September 6, 2026

Read-only evidence for the improvement plan. Counts have different grains: copies, card IDs, stints, event instances, and card/series aggregates. The saved SQL and TypeScript parser checks are inspectable below. No gameplay improvement has been experimentally established.


In [1]:
from pathlib import Path
import json
root = Path('/Users/ljmac/Desktop/OOTP Perfect Team')
audit_dir = root / 'Docs/audit-2026-09-06'
evidence = json.loads((audit_dir / 'evidence.json').read_text())
print('Audited:', evidence['auditedAt'])
print(json.dumps(evidence['rawFolders'], indent=2))


Audited: 2026-09-06T23:54:10.445Z
{
  "LJ Cards and Card Shop": {
    "csvFiles": 5,
    "bytes": 5818746
  },
  "League Data": {
    "csvFiles": 44,
    "bytes": 26998073
  },
  "Archive/Completed": {
    "csvFiles": 338,
    "bytes": 442643808
  },
  "Tourney Data": {
    "csvFiles": 77,
    "bytes": 148041084
  },
  "Inbox": {
    "csvFiles": 1,
    "bytes": 944515
  }
}


In [2]:
print(json.dumps(evidence['latestRawCards'], indent=2))
print(json.dumps(evidence['latestDumps'], indent=2))


{
  "shop": {
    "total": 4010,
    "ownedCards": 3436,
    "ownedCopies": 3500,
    "byTier": {
      "Silver": 675,
      "Iron": 1288,
      "Gold": 511,
      "Diamond": 346,
      "Bronze": 978,
      "Perfect": 212
    },
    "ownedByTier": {
      "Silver": 625,
      "Iron": 1217,
      "Gold": 407,
      "Diamond": 217,
      "Bronze": 923,
      "Perfect": 47
    },
    "withVariantListing": 2185,
    "tierBandsValid": true,
    "headerFields": 122,
    "dataFields": 123
  },
  "collection": {
    "total": 3500,
    "variants": 75,
    "active": 26,
    "hasActiveColumn": true
  },
  "matchRate": 1,
  "qualities": {
    "exact": 3425,
    "variant": 75
  }
}
[
  {
    "file": "pt27_tournaments_competitve_dump_20260831.csv",
    "source": "tournaments",
    "events": 8403,
    "uniqueIds": 8403,
    "from": "2026-03-13",
    "to": "2026-08-31",
    "personalEntries": 705
  },
  {
    "file": "pt27_drafts_competitve_dump_20260831.csv",
    "source": "drafts",
    "events": 583

In [3]:
db = evidence['database']
for name, value in db.items():
    if isinstance(value, int):
        print(name, value)
for name in ['uploads', 'latestCollection', 'latestOwnership', 'rules', 'observations', 'unmatchedObserved', 'personalResults', 'periods', 'dailyCoverage']:
    print(name, json.dumps(db[name], indent=2))


cards 4010
card_snapshots 11691
collection_cards 10445
tournaments 128
contexts 0
observed_card_stats 30321
series_meta 48
league_snapshots 47
league_stints 41375
rosters 0
results 0
daily_totals 350
my_results 1296
standings 0
uploads [
  {
    "kind": "collection",
    "uploads": 3,
    "latest_recorded_date": "2026-08-27T12:00:00.000Z"
  },
  {
    "kind": "dump",
    "uploads": 5,
    "latest_recorded_date": "2026-08-31T12:00:00.000Z"
  },
  {
    "kind": "league",
    "uploads": 47,
    "latest_recorded_date": "2026-09-06T12:00:00.000Z"
  },
  {
    "kind": "shop_list",
    "uploads": 3,
    "latest_recorded_date": "2026-08-27T12:00:00.000Z"
  }
]
latestCollection [
  {
    "match_quality": "exact",
    "is_variant": false,
    "rows": 3425,
    "distinct_cards": 3414
  },
  {
    "match_quality": "variant",
    "is_variant": true,
    "rows": 75,
    "distinct_cards": 75
  }
]
latestOwnership [
  {
    "owned_card_ids": 3436,
    "owned_copies": 3500
  }
]
rules [
  {
    "catalo

In [4]:
for row in db['snapshotCompleteness']:
    share = row['pitchers'] / row['rows'] if row['rows'] else 0
    if share < .15:
        print('Incomplete snapshot:', row, 'pitcher share:', round(share*100, 2))
print('Latest raw snapshot dates (completeness fallback may use an earlier date):')
print(json.dumps(db['latestLeague'], indent=2))


Incomplete snapshot: {'league': 'HD451', 'split': 'all', 'captured_on': '2026-08-30T05:00:00.000Z', 'rows': 473, 'pitchers': 4} pitcher share: 0.85
Latest raw snapshot dates (completeness fallback may use an earlier date):
[
  {
    "league": "HD450",
    "split": "all",
    "latest": "2026-09-06T05:00:00.000Z",
    "snapshots": 5
  },
  {
    "league": "HD450",
    "split": "vL",
    "latest": "2026-09-06T05:00:00.000Z",
    "snapshots": 4
  },
  {
    "league": "HD450",
    "split": "vR",
    "latest": "2026-09-06T05:00:00.000Z",
    "snapshots": 4
  },
  {
    "league": "HD451",
    "split": "all",
    "latest": "2026-08-30T05:00:00.000Z",
    "snapshots": 3
  },
  {
    "league": "HD451",
    "split": "vL",
    "latest": "2026-08-30T05:00:00.000Z",
    "snapshots": 2
  },
  {
    "league": "HD451",
    "split": "vR",
    "latest": "2026-08-30T05:00:00.000Z",
    "snapshots": 2
  },
  {
    "league": "HD452",
    "split": "all",
    "latest": "2026-09-06T05:00:00.000Z",
    "snapsho

## Queries and parser implementation

Every database query runs inside an explicit read-only transaction using one connection. The script below prints the full query definitions. The audit script also uses the app’s current shop, collection and dump parsers. Parser match coverage is not independent proof of identity accuracy.


In [5]:
for label, query in evidence['queries'].items():
    print(label + ':\n' + query + '\n')
print((audit_dir / 'audit.ts').read_text())


uploads:
select kind,count(*)::int as uploads,max(uploaded_at) as latest_recorded_date from uploads group by kind order by kind

latestCollection:
select match_quality,is_variant,count(*)::int as rows,count(distinct card_id)::int as distinct_cards from collection_cards where upload_id=(select max(id) from uploads where kind='collection') group by match_quality,is_variant

latestOwnership:
select count(*) filter(where owned>0)::int as owned_card_ids,sum(owned)::int as owned_copies from card_snapshots where upload_id=(select max(id) from uploads where kind='shop_list')

rules:
select count(*)::int as catalog,count(*) filter(where not retired)::int as current,count(*) filter(where restrictions->>'teamCap' is not null)::int as with_team_cap,count(*) filter(where restrictions->>'variantCap' is not null)::int as with_variant_cap,count(*) filter(where restrictions->>'slots' is not null)::int as with_slots,count(*) filter(where restrictions->>'cardTypes' is not null)::int as with_card_types fr

## Optional refresh

This function is defined but is not called by the notebook. Calling it reruns the read-only checks against the configured database and updates the local evidence file. Run with the existing app dependencies available. No credentials are printed.


In [6]:
def refresh_evidence():
    import subprocess
    return subprocess.run(
        ['node', '--env-file=.env.local', '--import', 'tsx', '../Docs/audit-2026-09-06/audit.ts'],
        cwd=root / 'web', check=True, capture_output=True, text=True
    )


## Interpretation

- The current ingest harness passed 43 executed checks but skipped collection and official standings fixture sections. The actual Aug 27 collection was tested separately.
- HD451 Aug 30 is incomplete and existing production selection falls back to Aug 16.
- 152 aggregate rows reference 34 card IDs absent from the catalog.
- Only event-level, rules-versioned history can support clean chronological validation after tournament refreshes.
- See NOTES.md for code-level findings, impact, suggested remediation and limits of verification.
